# Adijabatska teorema i adijabatsko računarstvo

Adijabatsko kvantno računarstvo (AQC) rešava optimizacione probleme tako što enkoduje rešenje kao **osnovno stanje** Hamiltonijana, a zatim polako evoluira kvantni sistem do tog stanja.

---

## 1. Adijabatska teorema

> Ako kvantni sistem počinje u osnovanom stanju $H(0)$ i Hamiltonijan se **dovoljno sporo** menja, sistem ostaje u trenutnom osnovanom stanju tokom cele evolucije.

Hamiltonijan se linearno interpolira između početnog $H_0$ (lako priprema osnovno stanje) i problemskog $H_P$ (osnovno stanje = rešenje):

$$H(s) = (1-s)\,H_0 + s\,H_P, \quad s = \frac{t}{T} \in [0,1]$$

### Uslov: energetski procep

Evolucija mora biti dovoljno spora u odnosu na **minimalni energetski procep** $\Delta_{\min}$:

$$T \gg \frac{1}{\Delta_{\min}^2}, \qquad \Delta(s) = E_1(s) - E_0(s)$$

Mali procep → spora evolucija → skupo računanje. Ovo je centralni izazov AQC.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Jednostavan 1-qubit primer: H0 = -sigma_x, HP = -sigma_z
s_vals = np.linspace(0, 1, 500)
E0, E1, gap = [], [], []

for s in s_vals:
    H = np.array([[-s, -(1-s)], [-(1-s), s]])
    evals = np.linalg.eigvalsh(H)
    E0.append(evals[0]); E1.append(evals[1])
    gap.append(evals[1] - evals[0])

E0, E1, gap = np.array(E0), np.array(E1), np.array(gap)
delta_min = gap.min()
s_gap = s_vals[np.argmin(gap)]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(s_vals, E0, 'b-', lw=2, label='$E_0(s)$ osnovno stanje')
ax1.plot(s_vals, E1, 'r--', lw=2, label='$E_1(s)$ pobuđeno stanje')
ax1.annotate('', xy=(s_gap, E1[np.argmin(gap)]), xytext=(s_gap, E0[np.argmin(gap)]),
             arrowprops=dict(arrowstyle='<->', color='green', lw=2))
ax1.text(s_gap + 0.03, (E0[np.argmin(gap)] + E1[np.argmin(gap)])/2,
         f'$\\Delta_{{min}}={delta_min:.2f}$', color='green', fontsize=11)
ax1.set_xlabel('$s = t/T$'); ax1.set_ylabel('Energija')
ax1.set_title('Energetski spektar'); ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(s_vals, gap, 'g-', lw=2)
ax2.axhline(delta_min, color='orange', linestyle='--', label=f'$\\Delta_{{min}}={delta_min:.2f}$')
ax2.set_xlabel('$s = t/T$'); ax2.set_ylabel('$\\Delta(s)$')
ax2.set_title('Energetski procep'); ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()
print(f"Δ_min = {delta_min:.4f}  →  T >> {1/delta_min**2:.1f}")

## 2. Veza sa QUBO

**QUBO** traži $\mathbf{x} \in \{0,1\}^n$ koje minimizuje:

$$f(\mathbf{x}) = \sum_i Q_{ii}\, x_i + \sum_{i < j} Q_{ij}\, x_i x_j$$

### Korak 1 — Supstitucija bita spinovim

Zamenom $x_i = \dfrac{1 - \hat{\sigma}_i^z}{2}$ svaki bit $x_i \in \{0,1\}$ postaje eigenvalue Paulijevog operatora $\hat{\sigma}_i^z \in \{+1, -1\}$:

$$x_i = 0 \;\longleftrightarrow\; \hat{\sigma}_i^z = +1 \qquad x_i = 1 \;\longleftrightarrow\; \hat{\sigma}_i^z = -1$$

### Korak 2 — Linearni član: dijagonala $Q_{ii}$

$$Q_{ii}\, x_i = Q_{ii}\cdot\frac{1-\hat{\sigma}_i^z}{2} = \frac{Q_{ii}}{2} - \frac{Q_{ii}}{2}\,\hat{\sigma}_i^z$$

Član $-\frac{Q_{ii}}{2}\hat{\sigma}_i^z$ je **lokalno magnetno polje** na qubitu $i$. Fizički:
- $Q_{ii} < 0$ (negativna dijagonala) → polje favorizuje $\hat\sigma_i^z = -1$, tj. $x_i = 1$
- $Q_{ii} > 0$ → polje favorizuje $\hat\sigma_i^z = +1$, tj. $x_i = 0$

### Korak 3 — Kvadratni član: van-dijagonala $Q_{ij}$

$$Q_{ij}\, x_i x_j = Q_{ij}\cdot\frac{1-\hat{\sigma}_i^z}{2}\cdot\frac{1-\hat{\sigma}_j^z}{2} = \frac{Q_{ij}}{4}\!\left(1 - \hat{\sigma}_i^z - \hat{\sigma}_j^z + \hat{\sigma}_i^z\hat{\sigma}_j^z\right)$$

Jedino **spinski korelacioni** član $\frac{Q_{ij}}{4}\,\hat{\sigma}_i^z\hat{\sigma}_j^z$ nije apsorpovan u lokalna polja. Fizički:
- $Q_{ij} > 0$ → **antiferromagnetno** kuplovanje: minimizuje energiju kada su $x_i \neq x_j$
- $Q_{ij} < 0$ → **feromagnetno** kuplovanje: minimizuje energiju kada su $x_i = x_j$

### Rezultujući Izing Hamiltonijan

Skupljanjem svih članova (konstante se odbacuju jer ne utiču na minimum):

$$\boxed{H_P = \sum_i h_i\, \hat{\sigma}_i^z + \sum_{i<j} J_{ij}\, \hat{\sigma}_i^z \hat{\sigma}_j^z}$$

gde su koeficijenti direktno određeni matricom $Q$:

$$h_i = -\frac{Q_{ii}}{2} - \sum_{j \neq i} \frac{Q_{ij}}{4}, \qquad J_{ij} = \frac{Q_{ij}}{4}$$

### Korak 4 — Adijabatska evolucija

Inicijalni Hamiltonijan $H_0 = -\sum_i \hat{\sigma}_i^x$ priprema ravnomernu superpoziciju svih $2^n$ konfiguracija kao osnovno stanje. Sistem zatim evoluira:

$$H(s) = (1-s)\,H_0 + s\,H_P, \quad s: 0 \to 1$$

Na kraju ($s = 1$), merenje u $\hat\sigma^z$ bazi daje $\mathbf{x}^*$ — rešenje originalnog QUBO problema.

| QUBO | Izing | Fizičko značenje |
|---|---|---|
| $Q_{ii} < 0$ | $h_i > 0$ | Lokalno polje favorizuje $x_i = 1$ |
| $Q_{ii} > 0$ | $h_i < 0$ | Lokalno polje favorizuje $x_i = 0$ |
| $Q_{ij} > 0$ | $J_{ij} > 0$ | Antiferromagnetno: $x_i \neq x_j$ |
| $Q_{ij} < 0$ | $J_{ij} < 0$ | Feromagnetno: $x_i = x_j$ |
| $\min f(\mathbf{x}^*)$ | $E_0$ osnovna energija | Rešenje = osnovno stanje |

## 3. Primer: QUBO → Izing za 2 bita

Posmatramo problem:

$$f(x_1, x_2) = -x_1 - x_2 + 2x_1 x_2, \qquad Q = \begin{pmatrix} -1 & 2 \\ 0 & -1 \end{pmatrix}$$

**Lokalna polja** (dijagonala + doprinos van-dijagonale):

$$h_1 = -\frac{Q_{11}}{2} - \frac{Q_{12}}{4} = -\frac{-1}{2} - \frac{2}{4} = \frac{1}{2} - \frac{1}{2} = 0$$

$$h_2 = -\frac{Q_{22}}{2} - \frac{Q_{12}}{4} = \frac{1}{2} - \frac{1}{2} = 0$$

**Kuplovanje** (van-dijagonala):

$$J_{12} = \frac{Q_{12}}{4} = \frac{2}{4} = \frac{1}{2}$$

Dakle, problemski Hamiltonijan je čisto antiferromagnetni Izing model bez lokalnih polja:

$$H_P = \frac{1}{2}\,\hat{\sigma}_1^z \hat{\sigma}_2^z$$

**Provera spektra** — u bazi $\{|00\rangle, |01\rangle, |10\rangle, |11\rangle\}$, eigenvalues $\hat\sigma_1^z\hat\sigma_2^z$ su:

| Stanje | $\hat\sigma_1^z$ | $\hat\sigma_2^z$ | $E = \frac{1}{2}\sigma_1^z\sigma_2^z$ | $f(x_1,x_2)$ |
|---|---|---|---|---|
| $\|00\rangle$ | $+1$ | $+1$ | $+\frac{1}{2}$ | $0$ |
| $\|01\rangle$ | $+1$ | $-1$ | $-\frac{1}{2}$ | $-1$ ← minimum |
| $\|10\rangle$ | $-1$ | $+1$ | $-\frac{1}{2}$ | $-1$ ← minimum |
| $\|11\rangle$ | $-1$ | $-1$ | $+\frac{1}{2}$ | $0$ |

Osnovna energija $E_0 = -\frac{1}{2}$ je **dvostruko degenerisana** i odgovara tačno minimumima $f = -1$ pri $(x_1, x_2) \in \{(0,1),\,(1,0)\}$. Antifero kuplovanje $J_{12} > 0$ fizički znači: sistem favorizuje antiparalelne spinove, što odgovara ekskluzivnom OR uslovu $x_1 \neq x_2$.

## Rezime

| Pojam | Suština |
|---|---|
| **Adijabatska teorema** | Sporo menjanje Hamiltonijana čuva osnovno stanje |
| **Energetski procep $\Delta_{\min}$** | Određuje potrebno vreme: $T \gg 1/\Delta_{\min}^2$ |
| **$H_0$** | Transverzalno polje $-\sum \hat\sigma^x$ — lako osnovno stanje |
| **$H_P$** | Izing Hamiltonijan koji enkoduje QUBO |
| **QUBO → Izing** | $x_i = (1 - \sigma_i^z)/2$ mapira bit na spin |
| **D-Wave** | Komercijalni AQC hardver (~5000 qubita, 15 mK) |

**Ograničenja:** Za NP-teške probleme $\Delta_{\min}$ može eksponencijalno opadati sa veličinom — kvadratno ubrzanje nije garantovano.

> **Sledeće:** QAOA — gate-bazirana aproksimacija adijabatske evolucije.